# Create DB for Evaluator Session

In [1]:
# load data
import pandas as pd
path_read = '../../data/raw/main/delegator_no_punish_2/all_apps_wide-2023-06-21.csv'
data = pd.read_csv(path_read)
data.shape

(30, 312)

In [2]:
data['post_decisions_no_punish.1.player.completed'].value_counts()

1    20
0    10
Name: post_decisions_no_punish.1.player.completed, dtype: int64

In [3]:
#remove dropouts
data_completed = data[data['post_decisions_no_punish.1.player.completed'] != 0]
data_completed.shape

(20, 312)

In [4]:
data_completed['outcome_for_eva'] = 0
data_completed.loc[(data_completed['post_decisions_no_punish.1.player.success']==1) | (data_completed['post_decisions_no_punish.1.player.algo_success']==1), 'outcome_for_eva'] = 1

C:\Users\Felix\AppData\Local\Temp\ipykernel_17140\2795418302.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_completed['outcome_for_eva'] = 0


In [5]:
#create excel file for evaluator session
df_for_eva = data_completed[['participant.code', 'outcome_for_eva', 'post_decisions_no_punish.1.player.delegation']] 
df_for_eva.columns = ['delegator_id', 'outcome', 'delegation']

In [6]:
#make integers
df_for_eva = df_for_eva.astype(
    {
        'outcome':'int',
        'delegation': 'int'
    }
)

In [7]:
#df_for_eva = df_for_eva.reset_index()
df_for_eva

,delegator_id,outcome,delegation
0,jdh5m9y6,1,1
1,5zbsqhlp,0,1
2,v7uljy90,0,1
4,nd2qi8z2,0,0
6,mj2d99wr,1,0
7,6t6hkmtr,0,1
9,v29kgi94,0,1
11,7o6vx9ld,0,1
13,er2awwqm,0,0
14,e0xudnw7,0,1


In [46]:
#write excel
#df_for_eva.to_excel('../../data/raw/main/delegator_no_punish_2/delegator_match.xlsx', index=False)

# Delegator Payoff

In [41]:
# load data
import pandas as pd
path_read = '../../data/raw/Test/all_apps_wide-2023-01-30.csv'
data = pd.read_csv(path_read)

In [42]:
#remove 
data_completed = data[data['questionnaire.1.player.completed'] != 0]

In [44]:
#load evaluator
eva_path_read = '../../data/raw/Test/all_apps_wide-2023-01-30_eva.csv'
eva_data = pd.read_csv(eva_path_read)

In [45]:
eva_data_completed = eva_data[eva_data['questionnaire_eva.1.player.completed'] != 0]

In [47]:
merged = data_completed.merge(eva_data_completed, left_on='participant.code', right_on='thank_you_eva.1.player.matched_delegator_id')

In [65]:
#calculate bonus payoffs delegator
merged['delegator_payoff'] = merged['session.config.flat_payment_delegator_x']
merged['evaluator_payoff'] = 0

for i in range(merged.shape[0]):
    #punishment payoff
    if merged['participant.delegation_x'][i] == 1:
        if merged['participant.overall_score_x'][i] >= merged['session.config.threshold_x'][i]:
            #del+good
            merged.loc[i, 'delegator_payoff'] -= merged['participant.punish_del_good_y'][i]
            merged.loc[i, 'evaluator_payoff'] += merged['session.config.good_outcome_y'][i]
            if merged['participant.punish_del_good_y'][i] == 0:
                merged.loc[i, 'evaluator_payoff'] += merged['session.config.punishment_cost_y'][i]
        elif merged['participant.overall_score_x'][i] < merged['session.config.threshold_x'][i]:
            #del+bad
            merged.loc[i, 'delegator_payoff'] -= merged['participant.punish_del_bad_y'][i]
            merged.loc[i, 'evaluator_payoff'] += merged['session.config.bad_outcome_y'][i]
            if merged['participant.punish_del_bad_y'][i] == 0:
                merged.loc[i, 'evaluator_payoff'] += merged['session.config.punishment_cost_y'][i]
    elif merged['participant.delegation_x'][i] == 0:
        #receive delegation cost
        merged.loc[i, 'delegator_payoff'] += merged['session.config.delegation_cost_x'][i]
        if merged['participant.overall_score_x'][i] >= merged['session.config.threshold_x'][i]:
            #nodel+good
            merged.loc[i, 'delegator_payoff'] -= merged['participant.punish_nodel_good_y'][i]
            merged.loc[i, 'evaluator_payoff'] += merged['session.config.good_outcome_y'][i]
            if merged['participant.punish_nodel_good_y'][i] == 0:
                merged.loc[i, 'evaluator_payoff'] += merged['session.config.punishment_cost_y'][i]
        elif merged['participant.overall_score_x'][i] < merged['session.config.threshold_x'][i]:
            #nodel+bad
            merged.loc[i, 'delegator_payoff'] -= merged['participant.punish_nodel_bad_y'][i]
            merged.loc[i, 'evaluator_payoff'] += merged['session.config.bad_outcome_y'][i]
            if merged['participant.punish_nodel_bad_y'][i] == 0:
                merged.loc[i, 'evaluator_payoff'] += merged['session.config.punishment_cost_y'][i]

          

In [108]:
import random
import numpy as np

#true performance in 11 bins
performance = pd.DataFrame(columns=['score', 'share'])

for i in range(data['session.config.n_predictions'][0] + 1):
    share = 100*sum(1 for x in data_completed['participant.overall_score'] if x==i)/len(data_completed['participant.overall_score'])
    performance.loc[i] = [i, share]
    
#performance belief payoffs
merged['belief_payoff_del'] = 0
merged['belief_payoff_eva'] = 0
merged['rand_del'] = np.nan
merged['rand_eva'] = np.nan

for i in range(merged.shape[0]):
    #choose random realization for delegator and evaluator
    merged.loc[i, 'rand_del'] = random.randint(0, 10)
    merged.loc[i, 'rand_eva'] = random.randint(0, 10)
    
    true_share_del = performance[performance['score']==merged['rand_del'][i]]['share']
    true_share_eva = performance[performance['score']==merged['rand_eva'][i]]['share']
    
    if (abs(int(true_share_del) - merged['performance_belief.1.player.difficulty' + str(int(merged['rand_del'][i])) + '_x'][i]) <= 5):
        merged.loc[i, 'belief_payoff_del'] += merged['session.config.others_belief_payment_x'][i]
    
    if (abs(int(true_share_eva) - merged['performance_belief.1.player.difficulty' + str(int(merged['rand_eva'][i])) + '_x'][i]) <= 5):
        merged.loc[i, 'belief_payoff_eva'] += merged['session.config.others_belief_payment_y'][i]
    
    

In [112]:
merged[['belief_payoff_del', 'rand_del', 'performance_belief.1.player.difficulty6_x']]

,belief_payoff_del,rand_del,performance_belief.1.player.difficulty6_x
0,1,8.0,19.0
1,1,8.0,0.0
2,0,0.0,100.0
3,1,6.0,2.0


In [94]:
int(performance[performance['score']==rand_del]['share'])

0